In [ ]:
import sys
sys.path.insert(0, "..")

import pyxdf
import os
import mne
mne.viz.set_browser_backend('matplotlib')  # or 'matplotlib' 'qt', etc.
import matplotlib.pyplot as plt
matplotlib.style.use('default')
%matplotlib inline
import numpy as np
from utils import closest_points_vector, split_video, create_mappings, create_events, create_mne

In [ ]:
ppg_data = pyxdf.load_xdf('../marker.xdf')


In [ ]:
ppg_data[0][1]

In [ ]:
for stream in ppg_data[0]:
    print(stream['info']['name'])


In [ ]:
# EXP_ROOT = "../exp_data"
# INPUT_ROOT = "input"
# OUTPUT_ROOT = "output"
# sub_id = 797337
# DATA_FILE = os.path.join(
#     EXP_ROOT, f"sub-{sub_id}", f"sub-{sub_id}_task-hearing_run-001.xdf"
# )
# RAW_VIDEO = os.path.join(EXP_ROOT, f"sub-{sub_id}", f"{sub_id}.avi")
data, header = pyxdf.load_xdf('../exp_data/01_Control/CTRL01/sub-617834/sub-617834_task-hearing_run-001.xdf')
marker_stream = next(
    stream for stream in data if stream["info"]["type"][0] == "Markers"
)
video_stream = next(stream for stream in data if stream["info"]["type"][0] == "Video")
eeg_stream = next(stream for stream in data if stream["info"]["type"][0] == "EEG")

In [ ]:
eeg_stream['time_stamps']

In [ ]:
video_stream['time_stamps']

In [ ]:
marker_stream['time_stamps']

In [ ]:
ch_labels = ['L1', 'L2', 'L4', 'L5', 'L7', 'L8', 'L9', 'L10',
             'R1', 'R2', 'R4', 'R5', 'R7', 'R8', 'R9', 'R10']
sampling_rate = 125

eeg_data = eeg_stream['time_series'].swapaxes(1, 0)
info = mne.create_info(
    ch_names=ch_labels, sfreq=sampling_rate, ch_types='eeg')
bindings = ['pmt', 'hlt', 'let', 'ast']
marker_data = np.array(marker_stream["time_series"]).squeeze()

marker_timestamps = marker_stream["time_stamps"]

eeg_insert_points = closest_points_vector(eeg_stream['time_stamps'], marker_timestamps)

marker_dict, id_binding, category_mapping = create_mappings(
    marker_data, bindings)
events = create_events(eeg_insert_points, marker_dict, marker_data)
bandpass = {"low": 1, "high": 50}
flat_voltage = 0.1
raw = create_mne(eeg_stream, events, id_binding,
                    bandpass=bandpass, flat_voltage=flat_voltage)

In [ ]:
raw.plot(scalings='auto')

In [ ]:
video_frames = video_stream["time_series"].squeeze()
insert_points = closest_points_vector(
    video_stream["time_stamps"], marker_stream["time_stamps"]
)
segments = []
for i in range(len(insert_points) - 1):
    start = insert_points[i]
    end = insert_points[i + 1] if i + 1 < len(insert_points) else -1
    segments.append(video_frames[start:end])

segments = segments
segment_arr = [(segments[i][0], segments[i][-1], marker_data[i]) for i in range(len(segments))]
print(segment_arr)

In [ ]:
split_video(RAW_VIDEO, segment_arr, OUTPUT_ROOT)

In [ ]:
for s in segment_arr:
    print(f"Start: {s[0]:.2f}, End: {s[1]:.2f}, Marker: {s[2]}, Duration: {(s[1]-s[0])/30:.2f}")